# Weather + Decision Layer — PV Soiling Triage Prototype

Combines: binary clean/dusty classifier output + live public weather data
(Open-Meteo, NASA POWER) -> rule-based recommendation (clean now / no
action / insufficient evidence). No inverter access, no dedicated sensors.

## 1. Weather data sources

Open-Meteo (rain forecast, soft signal only) and NASA POWER (irradiance).

In [ ]:
import requests

def get_rain_forecast(lat, lon, days=3):
    """Short-term rain forecast from Open-Meteo. Soft signal only - cited
    literature shows adherent soiling often survives rainfall."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat, "longitude": lon,
        "daily": "precipitation_sum,precipitation_probability_max",
        "forecast_days": days, "timezone": "auto",
    }
    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()
    daily = response.json()["daily"]
    precip_mm = daily["precipitation_sum"]
    precip_prob = daily["precipitation_probability_max"]
    # >=5mm AND >=50% probability = "meaningful rain event" - starting
    # heuristic, not a cited constant.
    rain_expected_soon = any(mm >= 5.0 and prob >= 50
                              for mm, prob in zip(precip_mm, precip_prob))
    return {"dates": daily["time"], "precip_mm": precip_mm,
            "precip_probability_max": precip_prob,
            "rain_expected_soon": rain_expected_soon}

In [ ]:
from datetime import date, timedelta

def get_irradiance_climatology(lat, lon, lookback_days=30):
    """Recent avg daily irradiance from NASA POWER - proxy for how much
    energy is at stake if soiling is left uncleaned."""
    end = date.today() - timedelta(days=2)  # POWER data lags ~1-2 days
    start = end - timedelta(days=lookback_days)
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "ALLSKY_SFC_SW_DWN", "community": "RE",
        "longitude": lon, "latitude": lat,
        "start": start.strftime("%Y%m%d"), "end": end.strftime("%Y%m%d"),
        "format": "JSON",
    }
    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()
    values = response.json()["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]
    daily_values = [v for v in values.values() if v != -999]  # -999 = missing
    if not daily_values:
        raise ValueError("No valid irradiance data for this location/range.")
    return {"avg_daily_irradiance_kwh_m2": round(sum(daily_values) / len(daily_values), 3),
            "n_days": len(daily_values)}

Smoke test against real Karachi coordinates.

In [ ]:
KARACHI_LAT, KARACHI_LON = 24.8607, 67.0011

rain_data = get_rain_forecast(KARACHI_LAT, KARACHI_LON)
irradiance_data = get_irradiance_climatology(KARACHI_LAT, KARACHI_LON)

print(rain_data)
print(irradiance_data)

{'dates': ['2026-09-04', '2026-09-05', '2026-09-06'], 'precip_mm': [1.1, 1.2, 0.8], 'precip_probability_max': [0, 6, 0], 'rain_expected_soon': False}
{'avg_daily_irradiance_kwh_m2': 4.654, 'n_days': 27}


## 2. Decision engine

Rule-based, not a calibrated optimizer. `decide()` outputs clean_now /
no_action / insufficient_evidence. See `src/decision_engine.py` in the repo
for the full version including `decide_with_cost_sensitivity()` (Week 3).

In [ ]:
from dataclasses import dataclass
from typing import Optional, Tuple

# Regional soiling-rate assumptions (cited, not invented)
SOILING_RATE_LOW = 0.003    # 0.3%/day  (NREL typical low end)
SOILING_RATE_MID = 0.008    # 0.8%/day  (Ghazi et al., Lahore - primary, regional placeholder not Karachi-specific)
SOILING_RATE_HIGH = 0.019   # 1.9%/day  (Cyprus arid seasonal average)

# Calibrated 25 Aug 2026 against val set (n=511): 96.3% coverage, 98.4% accuracy on covered subset
DEFAULT_CONFIDENCE_THRESHOLD = 0.70

@dataclass
class Recommendation:
    action: str
    reason: str
    est_daily_loss_pct_range: Optional[Tuple[float, float, float]]
    inputs_used: dict

def decide(classifier_label, classifier_confidence, irradiance_kwh_m2,
           rain_expected_soon, confidence_threshold=DEFAULT_CONFIDENCE_THRESHOLD):
    inputs_used = {"classifier_label": classifier_label,
                   "classifier_confidence": classifier_confidence,
                   "irradiance_kwh_m2": irradiance_kwh_m2,
                   "rain_expected_soon": rain_expected_soon,
                   "confidence_threshold": confidence_threshold}

    if classifier_confidence < confidence_threshold:
        return Recommendation("insufficient_evidence",
            f"Classifier confidence ({classifier_confidence:.2f}) is below "
            f"the {confidence_threshold:.2f} threshold. Retake the photo "
            f"with better lighting or a closer, more direct angle.",
            None, inputs_used)

    if classifier_label == "clean":
        return Recommendation("no_action",
            "Classifier detected a clean panel with sufficient confidence.",
            None, inputs_used)

    loss_range_pct = (SOILING_RATE_LOW * 100, SOILING_RATE_MID * 100, SOILING_RATE_HIGH * 100)
    rain_note = (" Rain is forecast soon, but adherent dust often survives "
                 "rainfall per the cited literature - this is not treated "
                 "as a reason to wait." if rain_expected_soon else "")
    reason = (f"Classifier detected dust with {classifier_confidence:.2f} confidence. "
              f"Estimated output loss ranges {loss_range_pct[0]:.2f}-{loss_range_pct[2]:.2f}%/day "
              f"(region-approximate estimate from Lahore, PK data - not Karachi-specific)." + rain_note)
    return Recommendation("clean_now", reason, loss_range_pct, inputs_used)

## 3. Confidence threshold calibration

Run against the real validation set to justify the threshold above, rather
than guessing one. Result (25 Aug 2026, val n=511): **0.70** -> 96.3%
coverage, 98.4% accuracy on the covered subset.

In [ ]:
!pip install ultralytics -q

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!kaggle datasets download -d hemanthsai7/solar-panel-dust-detection -p /content --unzip -q

Dataset URL: https://www.kaggle.com/datasets/hemanthsai7/solar-panel-dust-detection
License(s): CC0-1.0


In [ ]:
import random, shutil

random.seed(42)
SOURCE_DIR = "/content/Detect_solar_dust"
SPLIT_DIR = "/content/Detect_solar_dust_split"
CLASSES = ["Clean", "Dusty"]
VAL_FRACTION = 0.2

for cls in CLASSES:
    src_folder = os.path.join(SOURCE_DIR, cls)
    images = os.listdir(src_folder)
    random.shuffle(images)
    n_val = int(len(images) * VAL_FRACTION)
    val_images, train_images = images[:n_val], images[n_val:]
    for split_name, split_images in [("train", train_images), ("val", val_images)]:
        dest_folder = os.path.join(SPLIT_DIR, split_name, cls)
        os.makedirs(dest_folder, exist_ok=True)
        for img in split_images:
            shutil.copy(os.path.join(src_folder, img), os.path.join(dest_folder, img))

Upload `best.pt` from your local `models/` folder when prompted.

In [ ]:
from ultralytics import YOLO
from google.colab import files

uploaded = files.upload()  # select best.pt from models/ folder locally
model = YOLO("/content/best.pt")
class_names = model.names
print(class_names)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


Saving best.pt to best.pt
{0: 'Clean', 1: 'Dusty'}


In [ ]:
import numpy as np

true_labels, pred_labels, confidences = [], [], []

for cls in ["Clean", "Dusty"]:
    folder = f"/content/Detect_solar_dust_split/val/{cls}"
    results = model.predict(source=folder, verbose=False, stream=True)
    for r in results:
        pred_idx = int(r.probs.top1)
        true_labels.append(cls)
        pred_labels.append(class_names[pred_idx])
        confidences.append(float(r.probs.top1conf))

true_labels, pred_labels, confidences = np.array(true_labels), np.array(pred_labels), np.array(confidences)

print(f"{'Threshold':>10} | {'Coverage':>9} | {'Accuracy on covered':>20}")
print("-" * 46)
for threshold in np.arange(0.50, 1.00, 0.05):
    covered_mask = confidences >= threshold
    coverage = covered_mask.mean()
    acc = (true_labels[covered_mask] == pred_labels[covered_mask]).mean() if covered_mask.sum() > 0 else float("nan")
    print(f"{threshold:>10.2f} | {coverage:>8.1%} | {acc:>19.1%}")

 Threshold |  Coverage |  Accuracy on covered
----------------------------------------------
      0.50 |   100.0% |               97.3%
      0.55 |    99.2% |               97.6%
      0.60 |    97.8% |               98.2%
      0.65 |    97.1% |               98.4%
      0.70 |    96.3% |               98.4%
      0.75 |    94.3% |               99.0%
      0.80 |    93.2% |               98.9%
      0.85 |    91.8% |               98.9%
      0.90 |    89.8% |               98.9%
      0.95 |    84.0% |               99.1%


## 4. End-to-end pipeline test (real photo -> real weather -> real decision)

In [ ]:
import random as rnd

dusty_folder = "/content/Detect_solar_dust_split/val/Dusty"
sample_image = os.path.join(dusty_folder, rnd.choice(os.listdir(dusty_folder)))

result = model.predict(source=sample_image, verbose=False)[0]
pred_label = class_names[int(result.probs.top1)]
pred_confidence = float(result.probs.top1conf)

print(f"Image: {sample_image}")
print(f"Model prediction: {pred_label} (confidence {pred_confidence:.2f})")

rec = decide(classifier_label=pred_label.lower(), classifier_confidence=pred_confidence,
             irradiance_kwh_m2=irradiance_data["avg_daily_irradiance_kwh_m2"],
             rain_expected_soon=rain_data["rain_expected_soon"])

print(f"\nFinal recommendation: {rec.action}")
print(f"Reason: {rec.reason}")

Image: /content/Detect_solar_dust_split/val/Dusty/Imgdirty_584_1.jpg
Model prediction: Dusty (confidence 0.99)

Final recommendation: clean_now
Reason: Classifier detected dust with 0.99 confidence. Estimated output loss ranges 0.30-1.90%/day (region-approximate estimate from Lahore, PK data - not Karachi-specific).
